# Week 3 Assignment: Document Similarity

## Download 5 Online Documents

In [6]:
from pathlib import Path
from urllib.request import urlretrieve

urls = {
    "alice.txt": "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    "pride_and_prejudice.txt": "https://www.gutenberg.org/cache/epub/1342/pg1342.txt",
    "frankenstein.txt": "https://www.gutenberg.org/cache/epub/84/pg84.txt",
    "sherlock_holmes.txt": "https://www.gutenberg.org/cache/epub/1661/pg1661.txt",
    "moby_dick.txt": "https://www.gutenberg.org/cache/epub/2701/pg2701.txt"
}

folder = Path("week3_documents")
folder.mkdir(exist_ok=True)

for filename, url in urls.items():
    file_path = folder / filename
    if not file_path.exists():
        urlretrieve(url, file_path)

print(f"Downloaded {len(urls)} documents.")

Downloaded 5 documents.


## Read and Clean Documents

In [7]:
import re

stop_words = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for",
    "from", "has", "he", "in", "is", "it", "its", "of", "on",
    "that", "the", "to", "was", "were", "will", "with"
}

def clean_text(text):
    words = re.findall(r"[a-z]+", text.lower())
    return [word for word in words if word not in stop_words]

documents = {}
for file_path in sorted(folder.glob("*.txt")):
    text = file_path.read_text(encoding="utf-8", errors="ignore")
    documents[file_path.stem] = clean_text(text)

for name, words in documents.items():
    print(name, len(words), "words")

alice 21821 words
frankenstein 54871 words
moby_dick 155483 words
pride_and_prejudice 93177 words
sherlock_holmes 75433 words


## Create TF-IDF Vectors

In [8]:
import math
from collections import Counter

document_count = len(documents)
document_frequency = Counter()

for words in documents.values():
    document_frequency.update(set(words))

def create_tfidf(words):
    term_frequency = Counter(words)
    return {
        word: count * math.log(document_count / document_frequency[word])
        for word, count in term_frequency.items()
    }

tfidf_vectors = {
    name: create_tfidf(words)
    for name, words in documents.items()
}

## Calculate Cosine Similarity

In [9]:
def cosine_similarity(vector1, vector2):
    common_words = set(vector1) & set(vector2)
    dot_product = sum(vector1[word] * vector2[word] for word in common_words)
    length1 = math.sqrt(sum(value ** 2 for value in vector1.values()))
    length2 = math.sqrt(sum(value ** 2 for value in vector2.values()))
    return dot_product / (length1 * length2)

names = list(documents)
similarity_matrix = []

for name1 in names:
    row = []
    for name2 in names:
        score = cosine_similarity(tfidf_vectors[name1], tfidf_vectors[name2])
        row.append(round(score, 4))
    similarity_matrix.append(row)

## Similarity Results

In [10]:
import pandas as pd

similarity_table = pd.DataFrame(
    similarity_matrix,
    index=names,
    columns=names
)
similarity_table

,alice,frankenstein,moby_dick,pride_and_prejudice,sherlock_holmes
alice,1.0000,0.0030,0.0061,0.0018,0.0159
frankenstein,0.0030,1.0000,0.0724,0.0695,0.0240
moby_dick,0.0061,0.0724,1.0000,0.0081,0.0189
pride_and_prejudice,0.0018,0.0695,0.0081,1.0000,0.0203
sherlock_holmes,0.0159,0.0240,0.0189,0.0203,1.0000


## Rank Document Pairs

In [11]:
pairs = []

for i in range(len(names)):
    for j in range(i + 1, len(names)):
        pairs.append({
            "Document 1": names[i],
            "Document 2": names[j],
            "Similarity": similarity_matrix[i][j]
        })

ranking = pd.DataFrame(pairs).sort_values(
    "Similarity", ascending=False
).reset_index(drop=True)
ranking

,Document 1,Document 2,Similarity
0,frankenstein,moby_dick,0.0724
1,frankenstein,pride_and_prejudice,0.0695
2,frankenstein,sherlock_holmes,0.0240
3,pride_and_prejudice,sherlock_holmes,0.0203
4,moby_dick,sherlock_holmes,0.0189
5,alice,sherlock_holmes,0.0159
6,moby_dick,pride_and_prejudice,0.0081
7,alice,moby_dick,0.0061
8,alice,frankenstein,0.0030
9,alice,pride_and_prejudice,0.0018
